# 🧠 NS-ARC Slot-JEPA: 5-Phase Research-Grounded Training

Based on:
- **Slot Attention** (Locatello et al., NeurIPS 2020)
- **Slot-VAE** (Wang et al., ICML 2023)
- **Slot Mixture Module** (Vorobyov, ICLR 2024)
- **Causal-JEPA** (Object-Level Latent Interventions)

Principle: **Stabilize → Structure → Constrain → Reason**

| Phase | Name | Key Loss | Exit Condition |
|---|---|---|---|
| 1 | Slot Autoencoder | Recon (Focal) + KL | Recon < 2.0, distinct slot masks |
| 2 | GMM Competition | Phase1 + Orthogonality | Slot pair cosine sim < 0.3 |
| 3 | VICReg Geometry | Phase2 + Var + Cov | Latent Std between 0.8-1.2 |
| 4 | Causal Slot-JEPA | JEPA Pred (slot-level) | Prediction error decreasing |
| 5 | RbJEPA+EBC | Phase4 + Chamfer EBC | Convergence |

In [ ]:
import sys, os, subprocess

def run(cmd):
    print(f"Executing: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

KAGGLE_REPO_PATH = '/kaggle/working/Model-Jepa'
if os.path.exists(KAGGLE_REPO_PATH):
    print('✅ Kaggle detected. Pulling latest...')
    run(f'cd {KAGGLE_REPO_PATH} && git pull origin main || true')
    if KAGGLE_REPO_PATH not in sys.path:
        sys.path.insert(0, KAGGLE_REPO_PATH)
        print(f'✅ Added {KAGGLE_REPO_PATH} to sys.path')
else:
    sys.path.insert(0, os.path.abspath('.'))
    print('✅ Local mode.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

if torch.cuda.is_available(): DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

from modules.encoders import SlotTransformerEncoder, SlotDecoder
from modules.predictors import SlotJEPAPredictor
from modules.rule_encoders import RuleEncoder
from arc_data.rearc_dataset import ReARCDataset
from analysis.plot_utils import plot_reconstruction_dashboard, plot_slot_masks
from analysis.langevin_solver import LangevinGridSculptor

# ─────────────────────────────────────────────────────────────────────────────
# BASE CONFIG
# patch_size=2 → 30x30 grid → 15x15=225 patches (fine-grained semantics)
# slot_temperature=0.1 → hard slot competition (not fuzzy sharing)
# ─────────────────────────────────────────────────────────────────────────────
BASE_CFG = {
    'device': DEVICE,
    'input_dim': 64,
    'in_channels': 1,
    'patch_size': 2,          # CRITICAL: fine-grained for 30x30 ARC grids
    'hidden_dim': 256,
    'latent_dim': 128,
    'action_dim': 10,
    'num_slots': 8,            # Reduced from 16: fewer slots → easier to distinguish
    'slot_iters': 3,           # Original Slot Attention paper uses 3-7 iterations
    'slot_temperature': 0.1,  # Hard assignment (Slot Mixture Module insight)
    'encoder_layers': 4,
    'vocab_size': 10,
    'grid_size': 30,
    'focal_gamma': 2.0,
}

# Initialize all models
encoder = SlotTransformerEncoder(BASE_CFG).to(DEVICE)
decoder = SlotDecoder(BASE_CFG).to(DEVICE)
predictor = SlotJEPAPredictor(BASE_CFG).to(DEVICE)
rule_encoder = RuleEncoder(BASE_CFG).to(DEVICE)

# EMA Target Encoder (frozen copy)
target_encoder = SlotTransformerEncoder(BASE_CFG).to(DEVICE)
target_encoder.load_state_dict(encoder.state_dict())
for p in target_encoder.parameters(): p.requires_grad = False

print('✅ All modules initialized.')
print(f'  Encoder params: {sum(p.numel() for p in encoder.parameters()):,}')
print(f'  Decoder params: {sum(p.numel() for p in decoder.parameters()):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def kl_loss_slots(slot_mu, slot_logsigma):
    """
    KL divergence of the SLOT PRIOR towards N(0,1).
    Regularises the learned init distribution (slot_mu, slot_logsigma parameters).
    Acts as a leash preventing numerical explosion.  (Slot-VAE, Wang 2023)
    KL = -0.5 * (1 + log(sigma^2) - mu^2 - sigma^2)
    Expected behaviour:
      - Starts ~2.5 (because logsigma init = -3.0)
      - Decreases as the learned prior tightens toward N(0,1)
      - Should NOT go to 0 (that means mu=0, sigma=1 exactly — perfectly collapsed prior)
    """
    kl = -0.5 * (1 + 2 * slot_logsigma - slot_mu.pow(2) - (2 * slot_logsigma).exp())
    return kl.mean()


def slot_variance_loss(z):
    """
    Mild variance hinge on the OUTPUT latent z — forces slot vectors to be DIVERSE.
    This is the missing piece: KL regularises the PRIOR; this regularises the OUTPUT.

    z: [B, num_slots, latent_dim]

    Flattens to [B*num_slots, latent_dim] and computes std per latent dimension.
    Penalises any dimension whose std falls below 1.0.

    Lambda should be MILD in Phase 1 (e.g. 1.0, NOT 25.0 — that causes explosion).
    """
    B, S, D = z.shape
    z_flat = z.reshape(B * S, D)          # [B*S, D]
    std = torch.sqrt(z_flat.var(dim=0) + 1e-4)   # per-dimension std
    loss = torch.mean(F.relu(1.0 - std))          # hinge: penalise std < 1
    return loss, std.mean().item()


def slot_orthogonality_loss(z):
    """
    Penalize cosine similarity between different slots (Phase 2).
    Forces each slot to represent a different object.
    """
    z_norm = F.normalize(z, dim=-1)  # [B, S, D]
    sim = torch.bmm(z_norm, z_norm.transpose(1, 2))  # [B, S, S]
    S = sim.size(1)
    eye = torch.eye(S, device=z.device).unsqueeze(0)
    sim = sim * (1 - eye)
    return sim.abs().mean()


def off_diagonal(x):
    n, m = x.shape
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


def vicreg_loss(z, std_coeff=25.0, cov_coeff=1.0):
    """
    Full VICReg: Variance + Covariance (Phase 3 only).
    Applied ONLY after reconstruction is stable.
    """
    z = z - z.mean(dim=0)
    std = torch.sqrt(z.var(dim=0) + 1e-4)
    std_loss = torch.mean(F.relu(1.0 - std))
    cov = (z.T @ z) / (z.shape[0] - 1)
    cov_loss = off_diagonal(cov).pow_(2).sum() / z.shape[1]
    return std_coeff * std_loss + cov_coeff * cov_loss, std_loss, cov_loss


print('✅ Loss functions defined.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC PLOT HELPER
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('evaluation_reports/plots', exist_ok=True)

@torch.no_grad()
def run_diagnostics(encoder, decoder, dataset, cfg, epoch, phase_name, wb_run):
    encoder.eval()
    decoder.eval()

    # Use batch_size=8 for std computation — single sample gives misleading near-zero std
    b = dataset.sample(8)
    state = b['state'].to(cfg['device'])

    enc_out = encoder({'state': state})
    z = enc_out['latent']          # [8, num_slots, latent_dim]
    masks = enc_out['masks']       # [8, num_slots, H, W]

    dec_out = decoder({'latent': z, 'state': state})
    recon_logits = dec_out['reconstruction']  # [8, 10, H, W]
    recon_grid = recon_logits.argmax(dim=1).float()  # [8, H, W]

    # Latent stats — computed across batch * slots for honest std measurement
    B, S, D = z.shape
    z_flat = z.reshape(B * S, D).cpu().numpy()    # [B*S, D]
    latent_std_per_dim = z_flat.std(axis=0)        # std PER latent dimension
    latent_std_mean = float(latent_std_per_dim.mean())
    latent_std_min  = float(latent_std_per_dim.min())

    # Slot-pair cosine similarity (diversity metric)
    z_norm = z / (z.norm(dim=-1, keepdim=True) + 1e-8)
    sim = torch.bmm(z_norm, z_norm.transpose(1, 2))  # [B, S, S]
    mask_eye = torch.eye(S, device=z.device).unsqueeze(0).bool()
    sim_off = sim.masked_fill(mask_eye, 0.0)
    avg_slot_sim = float(sim_off.abs().mean().item())

    # Save reconstruction and slot mask plots for the FIRST sample only
    viz_path = f'evaluation_reports/plots/recon_{phase_name}_{epoch}.png'
    plot_reconstruction_dashboard(
        state[0, 0].cpu(),
        recon_grid[0].cpu(),
        torch.tensor(z_flat),
        epoch, viz_path
    )

    mask_path = f'evaluation_reports/plots/masks_{phase_name}_{epoch}.png'
    plot_slot_masks(masks[0:1], epoch, mask_path)

    wb_run.log({
        f'{phase_name}/Reconstruction':  wandb.Image(viz_path),
        f'{phase_name}/Slot_Masks':      wandb.Image(mask_path),
        f'{phase_name}/Latent_Std_Mean': latent_std_mean,
        f'{phase_name}/Latent_Std_Min':  latent_std_min,
        f'{phase_name}/Slot_Cosine_Sim': avg_slot_sim,
    })
    print(f'[Epoch {epoch:3d}] Std_Mean={latent_std_mean:.4f}  Std_Min={latent_std_min:.4f}  SlotSim={avg_slot_sim:.4f}')


print('✅ Diagnostic helper ready (batch-level std, slot cosine sim).')


---
## 🔷 Phase 1: Slot Autoencoder (Reconstruction + KL Leash)

**Paper basis:** Slot Attention (Locatello 2020) + Slot-VAE (Wang 2023)

**Why reconstruction first:**  
Without a pixel-level loss, there is ZERO gradient pulling individual slots toward individual objects.
The KL leash (λ=0.01) prevents numerical explosion by keeping slot vectors near N(0,1).

**Exit when:**
- `Phase_1/Recon_Loss` < 2.0  
- `Phase_1/Slot_Masks` shows distinct colored regions per slot

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Slot Autoencoder  (convergence-stable v3)
# Key additions vs v2:
#   + slot_variance_loss on z (fixes 'alpha shortcut' collapse)
#     The decoder can solve recon using alpha masks alone — z collapses.
#     This mild hinge (lambda=1.0) prevents that without causing explosion.
# ─────────────────────────────────────────────────────────────────────────────
dataset = ReARCDataset(data_path='/kaggle/working/Model-Jepa/arc_data/re-arc')
dataset._use_mock(3000)

PHASE1_EPOCHS = 100
PHASE1_BATCHES = 80
BATCH_SIZE = 16
LAMBDA_KL  = 0.001   # Leash on PRIOR params (prevents explosion)
LAMBDA_VAR = 1.0     # Hinge on OUTPUT z (prevents alpha-shortcut collapse)
WARMUP_EPOCHS = 5
TEMP_START = 1.0
TEMP_END   = 0.1

opt_p1 = torch.optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4, weight_decay=1e-4
)

import math
def warmup_cosine(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, PHASE1_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler_p1 = torch.optim.lr_scheduler.LambdaLR(opt_p1, warmup_cosine)

wb_run = wandb.init(project='NS-ARC-Slotted', name='Phase1-SlotAE-v3', reinit=True)
print('Starting Phase 1 (v3: alpha-shortcut fix)...')

for epoch in range(PHASE1_EPOCHS):
    encoder.train()
    decoder.train()

    tau = TEMP_START - (TEMP_START - TEMP_END) * (epoch / PHASE1_EPOCHS)
    encoder.temperature = tau
    decoder.config['focal_gamma'] = 0.0 if epoch < 20 else 2.0

    total_recon, total_kl, total_var, total_std = 0.0, 0.0, 0.0, 0.0

    for _ in range(PHASE1_BATCHES):
        batch = dataset.sample(BATCH_SIZE)
        states = batch['state'].to(DEVICE)

        enc_out = encoder({'state': states})
        z = enc_out['latent']              # [B, num_slots, 128]
        slot_mu = enc_out['slot_mu']
        slot_logsigma = enc_out['slot_logsigma']

        dec_out = decoder({'latent': z, 'state': states})
        loss_dict = decoder.loss({'state': states, 'latent': z}, dec_out)
        loss_recon = loss_dict['loss']

        # KL: regularises the PRIOR init distribution (leash against explosion)
        loss_kl = kl_loss_slots(slot_mu, slot_logsigma)

        # Variance hinge: regularises OUTPUT z (prevents alpha-shortcut collapse)
        loss_var, running_std = slot_variance_loss(z)

        loss = loss_recon + LAMBDA_KL * loss_kl + LAMBDA_VAR * loss_var

        opt_p1.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(decoder.parameters()), 1.0
        )
        opt_p1.step()

        total_recon += loss_recon.item()
        total_kl    += loss_kl.item()
        total_var   += loss_var.item()
        total_std   += running_std

    scheduler_p1.step()
    avg_recon = total_recon / PHASE1_BATCHES
    avg_kl    = total_kl    / PHASE1_BATCHES
    avg_var   = total_var   / PHASE1_BATCHES
    avg_std   = total_std   / PHASE1_BATCHES
    current_lr = opt_p1.param_groups[0]['lr']

    wb_run.log({
        'Phase_1/Recon_Loss':  avg_recon,
        'Phase_1/KL_Loss':     avg_kl,
        'Phase_1/Var_Loss':    avg_var,
        'Phase_1/Latent_Std':  avg_std,
        'Phase_1/Slot_Temp':   tau,
        'Phase_1/LR':          current_lr,
    })
    print(f'[P1 Epoch {epoch:3d}] Recon:{avg_recon:.4f}  KL:{avg_kl:.4f}  Var:{avg_var:.4f}  Std:{avg_std:.4f}  tau:{tau:.3f}')

    if epoch % 10 == 0:
        run_diagnostics(encoder, decoder, dataset, BASE_CFG, epoch, 'Phase_1', wb_run)

wb_run.finish()
print('✅ Phase 1 complete.')
print('   EXIT: Recon < 2.0 AND Latent_Std > 0.5 AND Slot_Cosine_Sim < 0.7')


---
## 🔷 Phase 2: Object-Centric Structuring (Orthogonality)

**Paper basis:** Slot Mixture Module (Vorobyov, ICLR 2024)

**Only run this if Phase 1 passed its exit condition.**

Adds slot orthogonality penalty to prevent multiple slots from encoding the same object.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Add Slot Orthogonality
# ─────────────────────────────────────────────────────────────────────────────
PHASE2_EPOCHS = 50
PHASE2_BATCHES = 80
LAMBDA_ORTH_P2 = 0.1

opt_p2 = torch.optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=2e-4, weight_decay=1e-4
)

wb_run = wandb.init(project='NS-ARC-Slotted', name='Phase2-Orthogonality', reinit=True)

print('Starting Phase 2: GMM-Style Slot Competition + Orthogonality...')
for epoch in range(PHASE2_EPOCHS):
    encoder.train()
    decoder.train()

    total_recon, total_kl, total_orth = 0.0, 0.0, 0.0

    for _ in range(PHASE2_BATCHES):
        batch = dataset.sample(BATCH_SIZE)
        states = batch['state'].to(DEVICE)

        enc_out = encoder({'state': states})
        z = enc_out['latent']
        slot_mu = enc_out['slot_mu']
        slot_logsigma = enc_out['slot_logsigma']

        dec_out = decoder({'latent': z, 'state': states})
        loss_dict = decoder.loss({'state': states, 'latent': z}, dec_out)

        loss_recon = loss_dict['loss']
        loss_kl = kl_loss_slots(slot_mu, slot_logsigma)
        loss_orth = slot_orthogonality_loss(z)

        loss = loss_recon + LAMBDA_KL * loss_kl + LAMBDA_ORTH_P2 * loss_orth

        opt_p2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(decoder.parameters()), 1.0
        )
        opt_p2.step()

        total_recon += loss_recon.item()
        total_kl += loss_kl.item()
        total_orth += loss_orth.item()

    avg_recon = total_recon / PHASE2_BATCHES
    avg_kl = total_kl / PHASE2_BATCHES
    avg_orth = total_orth / PHASE2_BATCHES

    wb_run.log({'Phase_2/Recon_Loss': avg_recon, 'Phase_2/KL_Loss': avg_kl, 'Phase_2/Orth_Loss': avg_orth})
    print(f'[Phase2 Epoch {epoch:3d}] Recon: {avg_recon:.4f}  KL: {avg_kl:.4f}  Orth: {avg_orth:.4f}')

    if epoch % 10 == 0:
        run_diagnostics(encoder, decoder, dataset, BASE_CFG, epoch, 'Phase_2', wb_run)

wb_run.finish()
print('✅ Phase 2 complete. Exit condition: Slot pair cosine sim < 0.3')

---
## 🔷 Phase 4: Causal Slot-JEPA (Object-Level Masking)

**Paper basis:** Causal-JEPA

Removes decoder. Masks **entire slots** (not pixels). Predicts masked slot latents from visible slots.
EMA target encoder provides stable learning signal (stop-gradient).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: Causal Slot-JEPA — Object-Level Slot Masking
# ─────────────────────────────────────────────────────────────────────────────

# Sync target encoder to current student state
target_encoder.load_state_dict(encoder.state_dict())
for p in target_encoder.parameters(): p.requires_grad = False

PHASE4_EPOCHS = 50
PHASE4_BATCHES = 100
NUM_SLOTS = BASE_CFG['num_slots']
MASK_SLOTS = max(1, NUM_SLOTS // 4)   # Mask ~25% of slots (Causal-JEPA recommendation)
EMA_MOMENTUM = 0.996

opt_p4 = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Also load real ARC data for p4
dataset.pairs = []
dataset._load()

wb_run = wandb.init(project='NS-ARC-Slotted', name='Phase4-CausalJEPA', reinit=True)

print('Starting Phase 4: Causal Slot-JEPA...')
print(f'  Masking {MASK_SLOTS}/{NUM_SLOTS} slots per sample (object-level, not pixel-level)')

for epoch in range(PHASE4_EPOCHS):
    encoder.train()
    predictor.train()
    target_encoder.eval()

    total_jepa = 0.0

    for _ in range(PHASE4_BATCHES):
        batch = dataset.sample(BATCH_SIZE)
        states = batch['state'].to(DEVICE)

        # Teacher: Encode full scene (stop-gradient)
        with torch.no_grad():
            target_z = target_encoder({'state': states})['latent']  # [B, S, D]

        # Object-Level Masking: mask entire slot indices for each sample
        masked_indices = torch.randperm(NUM_SLOTS)[:MASK_SLOTS]
        mask = torch.ones(BATCH_SIZE, NUM_SLOTS, 1, device=DEVICE)
        mask[:, masked_indices, :] = 0.0  # Zero out masked slots

        # Student: Encode full scene, then zero out masked slots
        enc_out = encoder({'state': states})
        z_full = enc_out['latent']  # [B, S, D]
        z_visible = z_full * mask   # Mask entire slot vectors

        # Predictor: predict missing slots from visible ones
        z_pred = predictor(z_visible)  # [B, S, D]

        # JEPA Loss: only on the MASKED slots (Causal-JEPA)
        target_masked = target_z[:, masked_indices, :]
        pred_masked = z_pred[:, masked_indices, :]
        loss_jepa = F.mse_loss(pred_masked, target_masked.detach())

        opt_p4.zero_grad()
        loss_jepa.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(predictor.parameters()), 1.0
        )
        opt_p4.step()

        # EMA update for target encoder
        target_encoder.update_ema(encoder, momentum=EMA_MOMENTUM)

        total_jepa += loss_jepa.item()

    avg_jepa = total_jepa / PHASE4_BATCHES
    wb_run.log({'Phase_4/JEPA_Loss': avg_jepa})
    print(f'[Phase4 Epoch {epoch:3d}] JEPA Loss: {avg_jepa:.4f}')

    if epoch % 10 == 0:
        sculptor = LangevinGridSculptor(target_encoder, DEVICE)
        b = dataset.sample(1)
        with torch.no_grad():
            enc_out = target_encoder({'state': b['state'].to(DEVICE)})
            z_c = enc_out['latent']
            masks = enc_out['masks']
        grid = sculptor.sculpt(z_c[0:1], (1, 1, 30, 30), steps=150)
        viz_path = f'evaluation_reports/plots/sculpt_p4_{epoch}.png'
        plot_reconstruction_dashboard(b['state'][0, 0], grid[0, 0], z_c[0].cpu(), epoch, viz_path)
        wb_run.log({'Phase_4/Sculpt': wandb.Image(viz_path)})
        mask_path = f'evaluation_reports/plots/masks_p4_{epoch}.png'
        plot_slot_masks(masks, epoch, mask_path)
        wb_run.log({'Phase_4/Slot_Masks': wandb.Image(mask_path)})

wb_run.finish()
print('✅ Phase 4 complete.')

---
## 🔷 Phase 5: RbJEPA + EBC (Rule-Informed)

Adds symbolic rule alignment (EBC Chamfer) to the Phase 4 Causal-JEPA framework.
Only run after Phase 4 shows a steadily decreasing JEPA loss.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5: RbJEPA + EBC Chamfer Loss
# ─────────────────────────────────────────────────────────────────────────────
# Reload mock data with symbolic rules for EBC
dataset.pairs = []
dataset._use_mock(3000)

PHASE5_EPOCHS = 50
PHASE5_BATCHES = 80
BETA_EBC = 0.1   # Low beta initially; increase manually as training stabilizes

opt_p5 = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()) + list(rule_encoder.parameters()),
    lr=5e-5, weight_decay=1e-4
)

wb_run = wandb.init(project='NS-ARC-Slotted', name='Phase5-RbJEPA-EBC', reinit=True)

print('Starting Phase 5: RbJEPA + EBC...')
for epoch in range(PHASE5_EPOCHS):
    encoder.train()
    predictor.train()
    rule_encoder.train()
    target_encoder.eval()

    total_jepa, total_ebc = 0.0, 0.0

    for _ in range(PHASE5_BATCHES):
        batch = dataset.sample(BATCH_SIZE)
        states = batch['state'].to(DEVICE)

        with torch.no_grad():
            target_z = target_encoder({'state': states})['latent']

        masked_indices = torch.randperm(NUM_SLOTS)[:MASK_SLOTS]
        mask = torch.ones(BATCH_SIZE, NUM_SLOTS, 1, device=DEVICE)
        mask[:, masked_indices, :] = 0.0

        enc_out = encoder({'state': states})
        z_full = enc_out['latent']
        z_visible = z_full * mask
        z_pred = predictor(z_visible)

        # JEPA Loss (Causal)
        loss_jepa = F.mse_loss(z_pred[:, masked_indices], target_z[:, masked_indices].detach())

        # EBC Chamfer Loss (Rule alignment)
        loss_ebc = torch.tensor(0.0, device=DEVICE)
        if 'rules' in batch:
            rules = batch['rules']
            z_rules_list = rule_encoder(rules)
            num_valid = 0
            for i in range(len(rules)):
                if len(rules[i]) > 0:
                    rule_vec = z_rules_list[i]    # [num_rules, latent_dim]
                    vis_vec = z_pred[i]            # [num_slots, latent_dim]
                    dists = torch.cdist(rule_vec.unsqueeze(0), vis_vec.unsqueeze(0))[0]
                    min_dists, _ = dists.min(dim=1)
                    loss_ebc = loss_ebc + min_dists.mean()
                    num_valid += 1
            if num_valid > 0:
                loss_ebc = loss_ebc / num_valid

        loss = loss_jepa + BETA_EBC * loss_ebc

        opt_p5.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(predictor.parameters()) + list(rule_encoder.parameters()), 1.0
        )
        opt_p5.step()
        target_encoder.update_ema(encoder, momentum=EMA_MOMENTUM)

        total_jepa += loss_jepa.item()
        total_ebc += loss_ebc.item()

    avg_jepa = total_jepa / PHASE5_BATCHES
    avg_ebc = total_ebc / PHASE5_BATCHES
    wb_run.log({'Phase_5/JEPA_Loss': avg_jepa, 'Phase_5/EBC_Loss': avg_ebc})
    print(f'[Phase5 Epoch {epoch:3d}] JEPA: {avg_jepa:.4f}  EBC: {avg_ebc:.4f}')

wb_run.finish()
print('✅ Phase 5 complete. Full training pipeline done.')